# Реакции пользователя

In [67]:
import numpy as np
import pandas as pd
import re
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit # для разделения с учетом групп
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from solution_utils import(read_dialogs, read_labels, get_groups, clean_normalize_dialog, calculate_metrics, SEED)

In [2]:
DATA_PATH = "hw1_train.jsonl"
LABELS = np.array(["positive", "negative", "neutral"])

dialogs = read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))
reactions = read_labels(DATA_PATH, "reaction")
labeled_dialogs = [d for d in dialogs if reactions.get(d.get('dialog_id')) is not None]
print("размеченных диалогов", len(labeled_dialogs))

диалогов: 10000
размеченных диалогов 3000


In [3]:
user_texts = [clean_normalize_dialog(d) for d in labeled_dialogs]
n_gr, groups_ids = get_groups(user_texts)
groups_ids = np.array(groups_ids)
print(f"групп: {n_gr}")
sizes = np.bincount(groups_ids)
print(f"групп >1: {(sizes > 1).sum()}")
print(f"макс размер: {sizes.max()}")
print(f"диалогов в группах >1: {sizes[sizes > 1].sum()}")

групп: 2978
групп >1: 17
макс размер: 4
диалогов в группах >1: 39


Разделяем на train, val и test с учетом баланса классов и группировки

In [24]:
X = np.array(user_texts)
y = np.array([reactions[d["dialog_id"]] for d in labeled_dialogs])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx_train_val, idx_test = next(gss.split(X, y, groups_ids))
gss = GroupShuffleSplit(n_splits=1, train_size=0.75, random_state=SEED)
idx_train, idx_val = next(gss.split(X[idx_train_val], y[idx_train_val], groups_ids[idx_train_val]))

X_train, y_train = X[idx_train_val[idx_train]], y[idx_train_val[idx_train]] # 60%
X_val, y_val = X[idx_train_val[idx_val]], y[idx_train_val[idx_val]] # 20%
X_test, y_test = X[idx_test], y[idx_test] # 20%

In [25]:
for name, labels_part in [
    ("train", y_train),
    ("validation", y_val),
    ("test", y_test),
]:
    print(f"\n{name}: {len(labels_part)} примеров")
    print(
        pd.Series(labels_part)
        .value_counts(normalize=True)
        .reindex(LABELS, fill_value=0)
        .round(3)
    )


train: 1804 примеров
positive    0.099
negative    0.318
neutral     0.583
Name: proportion, dtype: float64

validation: 599 примеров
positive    0.114
negative    0.302
neutral     0.584
Name: proportion, dtype: float64

test: 597 примеров
positive    0.094
negative    0.281
neutral     0.625
Name: proportion, dtype: float64


Получаем Tf-idf для всех данных

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # добавляем биграммы
    sublinear_tf=True,     # уменьшает скорость роста влияния слова от его частоты
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)
X_test_tfidf  = vectorizer.transform(X_test)

## Базовые модели

### Наивный Байес

Мы на семинаре использовали MultinomialNB, но сейчас больше подойдет ComplementNB, так как он специально спроектирован для несбалансированных данных, как у нас

In [28]:
alphas = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 0.9, 1.0, 1.5]
metrics_NB = []
for alpha in alphas:
    nb = ComplementNB(alpha=alpha)
    nb.fit(X_train_tfidf, y_train)
    y_pred = nb.predict(X_val_tfidf)
    metrics_NB.append(calculate_metrics(y_val, y_pred, LABELS))

metrics_NB_df = pd.DataFrame(metrics_NB, index=alphas).sort_values("macro-f1", ascending=False)
metrics_NB_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
0.005,0.631052,0.454941,0.450650,0.601589,"[[4, 19, 45], [7, 89, 85], [9, 56, 285]]","{'precision_positive': 0.2, 'recall_positive':..."
0.010,0.632721,0.453722,0.446006,0.601054,"[[3, 19, 46], [6, 91, 84], [7, 58, 285]]","{'precision_positive': 0.1875, 'recall_positiv..."
0.020,0.632721,0.446215,0.435940,0.595815,"[[2, 17, 49], [4, 87, 90], [6, 54, 290]]","{'precision_positive': 0.16666666666666666, 'r..."
0.001,0.624374,0.440564,0.431428,0.591465,"[[2, 18, 48], [7, 86, 88], [10, 54, 286]]","{'precision_positive': 0.10526315789473684, 'r..."
0.050,0.641068,0.441692,0.428998,0.595880,"[[1, 12, 55], [3, 81, 97], [4, 44, 302]]","{'precision_positive': 0.125, 'recall_positive..."
0.100,0.647746,0.431771,0.413350,0.588834,"[[0, 8, 60], [1, 70, 110], [0, 32, 318]]","{'precision_positive': 0.0, 'recall_positive':..."
0.200,0.646077,0.422815,0.403069,0.579534,"[[0, 5, 63], [0, 61, 120], [0, 24, 326]]","{'precision_positive': 0.0, 'recall_positive':..."
0.500,0.646077,0.416590,0.395157,0.572715,"[[0, 2, 66], [0, 54, 127], [0, 17, 333]]","{'precision_positive': 0.0, 'recall_positive':..."
1.000,0.622705,0.388140,0.355413,0.533789,"[[0, 2, 66], [0, 37, 144], [0, 14, 336]]","{'precision_positive': 0.0, 'recall_positive':..."
0.900,0.619366,0.387124,0.355322,0.532757,"[[0, 2, 66], [0, 38, 143], [0, 17, 333]]","{'precision_positive': 0.0, 'recall_positive':..."


По всем параметрам лучше всего alpha = !!!, выбираем его

In [29]:
alpha_opt = 0.005
nb = ComplementNB(alpha=alpha_opt).fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_val_tfidf)

## Логистическая регрессия

In [30]:
C_grid = [0.001, 0.01, 0.05, 0.1, 0.5, 1, 1.5, 2.0, 5.0]

metrics_logreg = []
for C in C_grid:
    log_reg = LogisticRegression(
        class_weight="balanced", # веса классов обратно пропорционально частоте
        max_iter=5000,
        C=C,
    )
    log_reg.fit(X_train_tfidf, y_train)
    y_pred = log_reg.predict(X_val_tfidf)
    metrics_logreg.append(calculate_metrics(y_val, y_pred, LABELS))

metrics_logreg_df = pd.DataFrame(metrics_logreg, index=C_grid).sort_values("macro-f1", ascending=False)
metrics_logreg_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
0.100,0.642738,0.557783,0.566799,0.639214,"[[27, 14, 27], [11, 95, 75], [18, 69, 263]]","{'precision_positive': 0.48214285714285715, 'r..."
0.001,0.639399,0.561606,0.566330,0.637991,"[[28, 14, 26], [13, 97, 71], [20, 72, 258]]","{'precision_positive': 0.45901639344262296, 'r..."
0.050,0.642738,0.559562,0.565786,0.640401,"[[27, 14, 27], [13, 97, 71], [19, 70, 261]]","{'precision_positive': 0.4576271186440678, 're..."
0.010,0.637730,0.560654,0.565243,0.636497,"[[28, 14, 26], [13, 97, 71], [20, 73, 257]]","{'precision_positive': 0.45901639344262296, 'r..."
0.500,0.651085,0.537069,0.554914,0.640521,"[[21, 13, 34], [7, 93, 81], [13, 61, 276]]","{'precision_positive': 0.5121951219512195, 're..."
1.500,0.656093,0.523239,0.545943,0.639787,"[[17, 14, 37], [3, 92, 86], [8, 58, 284]]","{'precision_positive': 0.6071428571428571, 're..."
1.000,0.656093,0.524128,0.545734,0.640379,"[[17, 14, 37], [4, 93, 84], [8, 59, 283]]","{'precision_positive': 0.5862068965517241, 're..."
2.000,0.657763,0.521131,0.544215,0.640308,"[[16, 14, 38], [2, 93, 86], [7, 58, 285]]","{'precision_positive': 0.64, 'recall_positive'..."
5.000,0.652755,0.508989,0.532925,0.632006,"[[15, 14, 39], [2, 87, 92], [5, 56, 289]]","{'precision_positive': 0.6818181818181818, 're..."


Выбираем C = 0.1

In [10]:
C_opt_logreg = 0.1
log_reg = LogisticRegression(
    class_weight="balanced", # веса классов обратно пропорционально частоте
    max_iter=5000,
    C=C_opt_logreg,
).fit(X_train_tfidf, y_train)
y_pred_logreg = log_reg.predict(X_val_tfidf)

## Метод опорных векторов

In [31]:
metrics_svc = []
for C in C_grid:
    svc = LinearSVC(
        class_weight="balanced", # веса классов обратно пропорционально частоте
        max_iter=5000,
        C=C,
    )
    svc.fit(X_train_tfidf, y_train)
    y_pred = svc.predict(X_val_tfidf)
    metrics_svc.append(calculate_metrics(y_val, y_pred, LABELS))

metrics_svc_df = pd.DataFrame(metrics_svc, index=C_grid).sort_values("macro-f1", ascending=False)
metrics_svc_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
2.000,0.669449,0.500046,0.518750,0.638892,"[[11, 14, 43], [2, 84, 95], [4, 40, 306]]","{'precision_positive': 0.6470588235294118, 're..."
1.500,0.667780,0.499983,0.518698,0.637969,"[[11, 13, 44], [2, 85, 94], [4, 42, 304]]","{'precision_positive': 0.6470588235294118, 're..."
1.000,0.667780,0.499983,0.518698,0.637969,"[[11, 13, 44], [2, 85, 94], [4, 42, 304]]","{'precision_positive': 0.6470588235294118, 're..."
5.000,0.662771,0.496237,0.514501,0.633284,"[[11, 14, 43], [2, 84, 95], [4, 44, 302]]","{'precision_positive': 0.6470588235294118, 're..."
0.500,0.657763,0.490712,0.510044,0.627037,"[[11, 12, 45], [1, 81, 99], [4, 44, 302]]","{'precision_positive': 0.6875, 'recall_positiv..."
0.100,0.652755,0.455265,0.458814,0.604838,"[[5, 9, 54], [0, 71, 110], [1, 34, 315]]","{'precision_positive': 0.8333333333333334, 're..."
0.050,0.664441,0.439019,0.424992,0.598686,"[[1, 8, 59], [0, 63, 118], [0, 16, 334]]","{'precision_positive': 1.0, 'recall_positive':..."
0.001,0.584307,0.333333,0.245873,0.430996,"[[0, 0, 68], [0, 0, 181], [0, 0, 350]]","{'precision_positive': 0.0, 'recall_positive':..."
0.010,0.584307,0.333333,0.245873,0.430996,"[[0, 0, 68], [0, 0, 181], [0, 0, 350]]","{'precision_positive': 0.0, 'recall_positive':..."


Лучший С = !!!

In [32]:
C_opt_svc = 2
svc = LinearSVC(
    class_weight="balanced", # веса классов обратно пропорционально частоте
    max_iter=5000,
    C=C_opt_svc,
).fit(X_train_tfidf, y_train)
y_pred_svc = svc.predict(X_val_tfidf)

In [68]:
metrics_models = []
metrics_models.append(calculate_metrics(y_val, y_pred_nb, LABELS))
metrics_models.append(calculate_metrics(y_val, y_pred_logreg, LABELS))
metrics_models.append(calculate_metrics(y_val, y_pred_svc, LABELS))

# baseline
dummy_most_frequent = DummyClassifier(strategy="most_frequent").fit(X_train_tfidf, y_train)
y_pred_dummy_mf = dummy_most_frequent.predict(X_val_tfidf)
metrics_models.append(calculate_metrics(y_val, y_pred_dummy_mf, LABELS))
dummy_stratified = DummyClassifier(strategy="stratified", random_state=0).fit(X_train_tfidf, y_train)
y_pred_dummy_st = dummy_stratified.predict(X_val_tfidf)
metrics_models.append(calculate_metrics(y_val, y_pred_dummy_st, LABELS))

metrics_models_df = pd.DataFrame(metrics_models, index=["NB", "LR", "SVC", "Dum_mf", "Dum_st"]).sort_values("macro-f1", ascending=False)
metrics_models_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
LR,0.642738,0.557783,0.566799,0.639214,"[[27, 14, 27], [11, 95, 75], [18, 69, 263]]","{'precision_positive': 0.48214285714285715, 'r..."
SVC,0.669449,0.500046,0.518750,0.638892,"[[11, 14, 43], [2, 84, 95], [4, 40, 306]]","{'precision_positive': 0.6470588235294118, 're..."
NB,0.631052,0.454941,0.450650,0.601589,"[[4, 19, 45], [7, 89, 85], [9, 56, 285]]","{'precision_positive': 0.2, 'recall_positive':..."
Dum_st,0.462437,0.347871,0.348146,0.465424,"[[8, 24, 36], [23, 59, 99], [47, 93, 210]]","{'precision_positive': 0.10256410256410256, 'r..."
Dum_mf,0.584307,0.333333,0.245873,0.430996,"[[0, 0, 68], [0, 0, 181], [0, 0, 350]]","{'precision_positive': 0.0, 'recall_positive':..."


Лучше всего себя показывает логистическая регрессия, берем ее за основу нашей модели и на ней проверяем гипотезы.

## Формируем "свою" модель на основе проверки гипотез о данных и параметрах

Изучим, как влияет выбор только первых/последних реплик на предсказания. Предположения: основная информация о реакции пользователя содержится в конце.

In [14]:
texts_by_variant = {
    "all": user_texts,
    "first": [
        clean_normalize_dialog(d, num=1, position='first')
        for d in labeled_dialogs
    ],
    "last": [
        clean_normalize_dialog(d, num=1, position='last')
        for d in labeled_dialogs
    ],
    "last_two": [
        clean_normalize_dialog(d, num=2, position='last')
        for d in labeled_dialogs
    ],
    "last_three": [
            clean_normalize_dialog(d, num=3, position='last')
            for d in labeled_dialogs
        ],
}

In [15]:
variant_results = []
variant_models = {}

for variant, texts in texts_by_variant.items():
    X_train_text = [texts[i] for i in idx_train_val[idx_train]]
    X_val_text   = [texts[i] for i in idx_train_val[idx_val]]

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True,
    )
    X_train_vec = vectorizer.fit_transform(X_train_text)
    X_val_vec   = vectorizer.transform(X_val_text)

    model = LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=5000,
        random_state=SEED,
    )
    model.fit(X_train_vec, y_train)
    pred = model.predict(X_val_vec)

    m = calculate_metrics(y_val, pred, LABELS)

    variant_results.append({
        "variant":         variant,
        "macro-f1":        m["macro-f1"],
        "weighted-f1":     m["weighted-f1"],
        "accuracy":        m["accuracy"],
        "balanced_acc":    m["balanced_accuracy"],
        "F1_positive":     m["per_class_metrics"]["f1_positive"],
        "F1_neutral":      m["per_class_metrics"]["f1_neutral"],
        "F1_negative":     m["per_class_metrics"]["f1_negative"],
    })

    variant_models[variant] = {
        "vectorizer": vectorizer,
        "model": model,
    }

variant_results_df = pd.DataFrame(variant_results).round(4).sort_values("macro-f1", ascending=False)

variant_results_df

,variant,macro-f1,weighted-f1,accuracy,balanced_acc,F1_positive,F1_neutral,F1_negative
3,last_two,0.5865,0.6544,0.6644,0.5627,0.4727,0.7513,0.5353
4,last_three,0.5817,0.6424,0.6461,0.5706,0.4839,0.7316,0.5296
0,all,0.5668,0.6392,0.6427,0.5578,0.4355,0.7357,0.5292
1,first,0.4995,0.5951,0.5993,0.4948,0.2903,0.6985,0.5097
2,last,0.4856,0.5771,0.5943,0.4659,0.3158,0.6965,0.4444


Ожидаемо, first — слабый, но, на удивление, last с одной репликой еще хуже. А вот 2 или 3 последние реплики уже лучше всего текста по большинству параметров. last_three выигрывает по balanced_accuracy и F1_positive, но остальные параметры лучше у last_two, пока берем его, но держим в голове и другой вариант.

Но нам этого недостаточно, плюс если мы совсем не будем учитывать остальной диалог, можем потерять длинные споры и недовольства в середине или (и так очень редкую) положительную реакцию. Попробуем просто добавить последним репликам веса.

Попробовали просто в hstack присоединить tfidf по последним строкам, умноженное на разные веса, но это почти ничего не дало. Не будем приводить здесь этот эксперимент, давайте попробуем добавить вес последним строкам иначе: отдельно обучаем 2 модели, одну на полных данных, вторую на последних репликах, и итоговые вероятности определяем как W * p_last + (1 - W) * p_full, p_last - вероятность, предсказанная по последним репликам, p_full - по всем

In [35]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=3)
X_tr_last2_tfidf = vectorizer.fit_transform([texts_by_variant.get("last_two")[i] for i in idx_train_val[idx_train]])
X_va_last2_tfidf = vectorizer.transform([texts_by_variant.get("last_two")[i] for i in idx_train_val[idx_val]])
log_reg  = LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000).fit(X_train_tfidf, y_train)
model_last2 = LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000).fit(X_tr_last2_tfidf, y_train)

results = []
weights = [0.0, 0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 1.0]
for W in weights:
    p_full  = log_reg.predict_proba(X_val_tfidf)
    p_last2 = model_last2.predict_proba(X_va_last2_tfidf)
    p_res = W * p_last2 + (1 - W) * p_full # взвешенное среднее вероятностей
    pred = log_reg.classes_[p_res.argmax(axis=1)]
    m = calculate_metrics(y_val, pred, LABELS)
    results.append({
            "macro-f1":        m["macro-f1"],
            "weighted-f1":     m["weighted-f1"],
            "accuracy":        m["accuracy"],
            "balanced_acc":    m["balanced_accuracy"],
            "F1_positive":     m["per_class_metrics"]["f1_positive"],
            "F1_neutral":      m["per_class_metrics"]["f1_neutral"],
            "F1_negative":     m["per_class_metrics"]["f1_negative"],
        })

results = pd.DataFrame(results, index=weights).round(4).sort_values("macro-f1", ascending=False)
results

,macro-f1,weighted-f1,accuracy,balanced_acc,F1_positive,F1_neutral,F1_negative
0.55,0.6046,0.6670,0.6745,0.5838,0.5043,0.7588,0.5507
0.60,0.6035,0.6656,0.6728,0.5828,0.5043,0.7571,0.5491
0.65,0.5993,0.6612,0.6678,0.5800,0.5000,0.7520,0.5460
0.40,0.5988,0.6635,0.6711,0.5779,0.4912,0.7561,0.5491
0.50,0.5975,0.6627,0.6711,0.5761,0.4912,0.7574,0.5439
0.70,0.5956,0.6578,0.6644,0.5772,0.4957,0.7493,0.5418
0.20,0.5875,0.6501,0.6561,0.5706,0.4915,0.7452,0.5257
0.30,0.5873,0.6506,0.6578,0.5676,0.4870,0.7446,0.5303
1.00,0.5807,0.6375,0.6427,0.5674,0.5041,0.7308,0.5072
0.00,0.5668,0.6392,0.6427,0.5578,0.4355,0.7357,0.5292


По результатам валидации учший вес W = !!! для 2 последних строк. Причем вес 0 для последних строк - худший в данном эксперименте, и прирост идет по всем рассмотренным параметрам (кроме F1_negative, оно такое же), особенно для нашего самого редкого позитивного класса! 

In [36]:
WEIGHT = 0.55

#### ??? Разные уровни очистки ???

Мы взяли самый общий уровень очистки данных - только реплики пользователя, нормализованные, без стоп-слов и очищенные от кода. Может быть, что-то из этого было лишним?

In [17]:
import re
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
from solution_utils import delete_code_lines, normalize

_stem = SnowballStemmer("english")
_WORD = re.compile(r"[^\W\d_]+")

def normalize_no_stop_words(text: str) -> list[str]:
    """БЕЗ удаления стоп-слов."""
    words = _WORD.findall(text.lower())
    return [_stem.stem(word) for word in words if len(word) >= 2]

def processing_dialog(dialog, include_assistant, clean_code, drop_stopwords: bool):
    """почти clean_normalize_dialog но с опциональным удалением стоп-слов"""
    if include_assistant:
        text = [turn["text"] for turn in dialog["turns"]]
    else:
        text = [turn["text"] for turn in dialog["turns"] if turn["role"] == "user"]
    
    text = "\n".join(text)
    if clean_code:
        text = delete_code_lines(text)
    if drop_stopwords:
        clean_norm_text = " ".join(normalize(text))
    else:
        clean_norm_text = " ".join(normalize_no_stop_words(text))
    return clean_norm_text


In [18]:
variants = [
    ("basic", False, True, True),
    ("no clean_code", False, False, True),
    ("no stopwords", False, True,  False),
    ("with assistant", True, True, True),
    ("with assistant, no clean", True, False, True),
]

rows = []
for name, inc_a, clean, drop in variants:
    texts = [processing_dialog(d, inc_a, clean, drop) for d in labeled_dialogs]

    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    X_tr = vec.fit_transform([texts[i] for i in idx_train_val[idx_train]])
    X_va = vec.transform([texts[i] for i in idx_train_val[idx_val]])

    l_r = LogisticRegression(C=0.1, class_weight="balanced",
                             max_iter=5000, random_state=SEED).fit(X_tr, y_train)
    pred = l_r.predict(X_va)

    m = calculate_metrics(y_val, pred, LABELS)
    rows.append({
        "variant":       name,
        "macro-f1":      m["macro-f1"],
        "weighted-f1":   m["weighted-f1"],
        "accuracy":      m["accuracy"],
        "balanced_acc":  m["balanced_accuracy"],
        "F1_positive":   m["per_class_metrics"]["f1_positive"],
        "F1_neutral":    m["per_class_metrics"]["f1_neutral"],
        "F1_negative":   m["per_class_metrics"]["f1_negative"],
    })

variants_df = pd.DataFrame(rows).round(4).sort_values("macro-f1", ascending=False)
variants_df

,variant,macro-f1,weighted-f1,accuracy,balanced_acc,F1_positive,F1_neutral,F1_negative
1,no clean_code,0.5777,0.6534,0.6578,0.5681,0.4355,0.7510,0.5465
2,no stopwords,0.5726,0.6510,0.6477,0.5865,0.4224,0.7500,0.5455
0,basic,0.5668,0.6392,0.6427,0.5578,0.4355,0.7357,0.5292
3,with assistant,0.5665,0.6574,0.6678,0.5494,0.3654,0.7541,0.5801
4,"with assistant, no clean",0.5665,0.6574,0.6678,0.5494,0.3654,0.7541,0.5801


In [19]:
#TODO добавить в качестве признака количество важных знаков препинания !!! ???
# проверить влияние данных вроде длины реплик, попробовать разные уровни очистки текста и нормализации
# добавить в работу с кодом выкидывание по ''' !!!!!!

## Еще эксперименты с дополнительными метриками

Теперь попробуем добавить еще параметры, описывающие текст, и посмотреть, станет ли лучше.
В качестве новых параметров будем рассматривать:

1 эксперимент:
длина текста в символах
длина текста в словах
длина последних 2 реплик в символах
длина последних 2 реплик в словах
Предположение: не ожидаем особого прироста метрик

2 эксперимент:
количество ! в тексте
количество ? в тексте
количество идущих подряд знаков !!, ?? или !?
Предположение: может улучшить нахождение негативных реакций

3 эксперимент:
для заглавных букв в тексте
наличие слов из 2+ заглавных букв
Предположение: может улучшить нахождение негативных реакций

In [62]:
metrix_features = []
m = calculate_metrics(y_val, y_pred_logreg, labels=LABELS)
metrix_features.append({
            "macro-f1":      m["macro-f1"],
            "weighted-f1":   m["weighted-f1"],
            "accuracy":      m["accuracy"],
            "balanced_acc":  m["balanced_accuracy"],
            "F1_positive":   m["per_class_metrics"]["f1_positive"],
            "F1_neutral":    m["per_class_metrics"]["f1_neutral"],
            "F1_negative":   m["per_class_metrics"]["f1_negative"],
        })

In [63]:
def features_lens(dialog):
    full_user = "\n".join(t["text"] for t in dialog["turns"] if t["role"] == "user")
    last2_user = "\n".join([t["text"] for t in dialog["turns"] if t["role"] == "user"][-2:])
    return [
        len(full_user),
        len(full_user.split()),
        len(last2_user),
        len(last2_user.split()),
    ]

EXCL = re.compile(r"!")
QUEST = re.compile(r"\?")
REPEAT = re.compile(r"!!|\?\?|!\?|\?!")

def features_puck(dialog):
    text = "\n".join(t["text"] for t in dialog["turns"] if t["role"] == "user")
    return [
        len(EXCL.findall(text)),
        len(QUEST.findall(text)),
        len(REPEAT.findall(text)),
    ]

CAPS_WORD = re.compile(r"\b[A-ZА-Я]{2,}\b")

def features_caps(dialog):
    text = "\n".join(t["text"] for t in dialog["turns"] if t["role"] == "user")
    letters = [l for l in text if l.isalpha()]
    if not letters:
        caps_ratio = 0
    else:  
        caps_ratio = sum(c.isupper() for c in letters) / len(letters)
    return [
        caps_ratio,
        int(bool(CAPS_WORD.search(text)))
    ]

In [64]:
def metrix_with_features(func):
    matrix_tr = csr_matrix([func(labeled_dialogs[i]) for i in idx_train_val[idx_train]])
    matrix_va = csr_matrix([func(labeled_dialogs[i]) for i in idx_train_val[idx_val]])

    # надо бы как-то нормировать наши признаки перед добавкой к общей матрице
    scaler = StandardScaler()
    F_tr_scaled = scaler.fit_transform(matrix_tr.toarray())
    F_va_scaled = scaler.transform(matrix_va.toarray())

    X_tr = hstack([X_train_tfidf, csr_matrix(F_tr_scaled)])
    X_va = hstack([X_val_tfidf, csr_matrix(F_va_scaled)])

    model = LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000, random_state=SEED).fit(X_tr, y_train)
    pred = model.predict(X_va)
    m = calculate_metrics(y_val, pred, labels=LABELS)
    return{
            "macro-f1":      m["macro-f1"],
            "weighted-f1":   m["weighted-f1"],
            "accuracy":      m["accuracy"],
            "balanced_acc":  m["balanced_accuracy"],
            "F1_positive":   m["per_class_metrics"]["f1_positive"],
            "F1_neutral":    m["per_class_metrics"]["f1_neutral"],
            "F1_negative":   m["per_class_metrics"]["f1_negative"],
        }

In [65]:
metrix_features.append(metrix_with_features(features_lens))
metrix_features.append(metrix_with_features(features_puck))
metrix_features.append(metrix_with_features(features_caps))
metrix_features = pd.DataFrame(metrix_features, index=["base", "lens", "punct", "caps"]).sort_values("macro-f1", ascending=False)
metrix_features

,macro-f1,weighted-f1,accuracy,balanced_acc,F1_positive,F1_neutral,F1_negative
base,0.566799,0.639214,0.642738,0.557783,0.435484,0.735664,0.529248
punct,0.526312,0.610236,0.614357,0.523023,0.375940,0.723227,0.479769
lens,0.512437,0.584838,0.562604,0.570712,0.368852,0.673046,0.495413
caps,0.471541,0.549949,0.530885,0.500487,0.306122,0.638844,0.469657


Вывод: никакой видимой пользы от новых параметров не обнаружено.